In [17]:
import json
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")

collection = client["activity6"]["students"]

# Delete existing data from database
collection.delete_many({})

# Insert data from file into database
with open("People.json") as file:
    data = json.load(file)
    collection.insert_many(data["students"])

print("Student full names over age 25:")
for row in collection.find({"age": {"$gt": 25}}, {"_id": 0, "fullName": 1}):
    print(row)

print("Student IDs with no middle names:")
for row in collection.find(
    {"$or": [
        {"fullName.other": {"$exists": False}},
        {"fullName.other": {"$exists": True, "$eq":[]}},
        {"fullName.other": {"$not": {"$elemMatch": {"$ne": None}}}}
    ]},
    {"_id": 0, "id": 1}
):
    print(row)

print("Men and women not living in Tokyo:")
for row in collection.aggregate([
    {"$match": {"city": {"$ne": "Tokyo"}}},
    {"$group": {"_id": "$fullName.title", "count": {"$sum": 1}}}
]):
    print(row)


Student full names over age 25:
{'fullName': {'title': 'Mrs', 'first': 'Lisa', 'surname': 'Penny', 'other': ['Melanie']}}
{'fullName': {'title': 'Mr', 'first': 'Lorenzo', 'surname': 'Dubois', 'other': ['Ruelle', 'Garlen']}}
{'fullName': {'title': 'Mr', 'first': 'Tanveer', 'surname': 'Patel', 'other': ['Vihaan']}}
Student IDs with no middle names:
{'id': 546854}
Men and women not living in Tokyo:
{'_id': 'Miss', 'count': 1}
{'_id': 'Mrs', 'count': 1}
{'_id': 'Mr', 'count': 2}
